# Qwen2.5-1.5B-Instruct SQL Fine-tuning with RAG and Validator
**Runtime required:** GPU (T4 or better)

**What this notebook does:**
1. Fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` on `train.jsonl` using QLoRA
2. Saves adapter to Google Drive
3. RAG layer: pass schema at query time — supports **multi-table schemas**
4. **Validates every SQL**: syntax + schema + semantic (blocks free text / natural language)
5. **Auto-corrects failures**: re-prompts with specific error, up to `max_retries` times
6. **Strict output guard**: rejects any response that is not valid SQL

## Install Dependencies

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl
!pip install -q sqlglot   # SQL AST parser used by the validator

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ No GPU! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Upload Training Data

Upload your **`train_qwen.jsonl`** file — this is the Qwen2.5 chat-template format:

```
<|im_start|>system
You are a strict SQL generator...<|im_end|>
<|im_start|>user
Use the schema below to write SQL.
Schema: table(cols) Question: ...<|im_end|>
<|im_start|>assistant
SELECT ...;<|im_end|>
```



In [ ]:
from google.colab import files
import json

print("Upload your train_qwen.jsonl file:")
uploaded = files.upload()
TRAIN_FILE = list(uploaded.keys())[0]
print(f"\nUploaded: {TRAIN_FILE}")

# Validate format
errors = []
count = 0
with open(TRAIN_FILE) as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            text = obj.get("text", "")
            if "<|im_start|>system" not in text or "<|im_start|>assistant" not in text:
                errors.append(f"Line {i+1}: missing Qwen2.5 chat tokens")
            count += 1
        except Exception as e:
            errors.append(f"Line {i+1}: {e}")

# Preview first example
print("\n--- First example preview ---")
with open(TRAIN_FILE) as f:
    ex = json.loads(f.readline())
print(ex["text"][:400] + " ...")


## Configuration

In [ ]:
# =================================================
#  Edit these settings based on your preference
# =================================================

model_name    = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR    = "./qwen25-sql-lora"
ADAPTER_DIR   = "./qwen25_sql_adapter"
DRIVE_DIR     = "/content/drive/MyDrive/qwen25_sql_adapter"

NUM_EPOCHS    = 14
BATCH_SIZE    = 2
GRAD_ACCUM    = 4      
LEARNING_RATE = 2e-4

MAX_SEQ_LEN   = 1024

print("Config ")
print(f"  Model          : {model_name}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Epochs         : {NUM_EPOCHS}")
print(f"  Max seq len    : {MAX_SEQ_LEN}")

## Load Model and Tokenizer (4-bit QLoRA)
QLoRA compresses the model weights from 16-bit → 4-bit, cutting VRAM by ~4x.
Qwen2.5-1.5B-Instruct fits comfortably on a free Colab T4 in 4-bit.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# 4-bit quantization config (NF4 = best quality at 4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,   # Compute in float16 for speed
    bnb_4bit_use_double_quant=True,         # Quantize the quantization constants too
    bnb_4bit_quant_type="nf4"               # NormalFloat4: best for normally-distributed weights
)

# Tokenizer
print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)

tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"  pad_token : {repr(tokenizer.pad_token)}")
print(f"  eos_token : {repr(tokenizer.eos_token)}")
print(f"  vocab size: {tokenizer.vocab_size:,}")

# Model in 4-bit
print("⏳ Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded")

## Load & Tokenize Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=TRAIN_FILE)
print(f"Loaded {len(dataset['train'])} training examples")

def tokenize(example):
    outputs = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    )
    # Labels = input_ids so the model learns to predict every token
    outputs["labels"] = outputs["input_ids"].copy()
    # Qwen may add token_type_ids — drop to avoid model.generate() errors
    outputs.pop("token_type_ids", None)
    return outputs

tokenized_dataset = dataset.map(tokenize, batched=True)
print(f"Tokenized. Columns: {tokenized_dataset['train'].column_names}")

## Apply LoRA
LoRA freezes the base model and adds small trainable rank-decomposition matrices.

**Qwen2.5 LoRA targets** — the attention projection names 
- Qwen2.5: same names, **plus** `gate_proj`, `up_proj`, `down_proj` (MLP projections)

Including the MLP projections gives Qwen2.5 a modest quality boost at minimal VRAM cost.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training (enables gradient checkpointing etc.)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    # Qwen2.5 attention projections + MLP projections
    # Adding gate/up/down_proj improves SQL generation quality
    # vs attention-only LoRA for instruction-following tasks.
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Fine-tune

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
    # Warmup over first 6% of steps for stability
    warmup_ratio=0.06,
    # Mild regularisation to prevent overfitting
    weight_decay=0.01,
    # Required for gradient checkpointing with QLoRA
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",   # memory-efficient optimizer for QLoRA
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset["train"],
    args=training_args,
)

print("Starting fine-tuning...")
trainer.train()
print("Training complete!")

## 💾 Step 8 — Save Adapter

In [ ]:
import shutil, os

# Save locally
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved locally: {ADAPTER_DIR}")

# Save to Google Drive (recommended — Colab sessions reset!)
from google.colab import drive
drive.mount("/content/drive")

if os.path.exists(DRIVE_DIR):
    shutil.rmtree(DRIVE_DIR)
shutil.copytree(ADAPTER_DIR, DRIVE_DIR)
print(f"Adapter backed up to Drive: {DRIVE_DIR}")

## Load Fine-tuned Model for Inference
This cell reloads the trained adapter on top of the base model.
If starting a **new session**, run this cell (and the install cell) — no need to retrain.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# If loading from Drive in a new session, change ADAPTER_DIR:
# ADAPTER_DIR = "/content/drive/MyDrive/qwen25_sql_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("⏳ Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("⏳ Loading fine-tuned adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
tokenizer.padding_side = "left"   # left-padding for generation (Qwen convention)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✅ Model ready for inference!")

## RAG Pipeline + SQL Validator + Auto-Corrector

Every query goes through a 3-stage pipeline:

**Stage 1 — GENERATE**: Qwen2.5 produces raw SQL from the prompt

**Stage 2 — VALIDATE** (4 checks):
- **Syntax**: sqlglot parse tree — catches malformed SQL, non-SQL output, garbage tokens, and free-text responses
- **Free-text guard**: rejects output that looks like natural language
- **Schema**: multi-table aware — checks table names, generates JOIN hints
- **Semantic**: catches DML for SELECT questions; LIMIT without intent

**Stage 3 — AUTO-CORRECT**: if validation fails, the specific error is appended and the model retries (up to `max_retries=4` times).

**Qwen2.5 prompt format** uses the chat template:
```
<|im_start|>system\nYou are a strict SQL generator...\n<|im_end|>
<|im_start|>user\nSchema: ... Question: ...<|im_end|>
<|im_start|>assistant
```


In [ ]:
import sqlglot
import sqlglot.expressions as exp
import re


# ================================================================
# SCHEMA PARSING  (multi-table aware)
# ================================================================
def parse_schema(schema):
    """
    Parse one or more table definitions.
    'actor(actor_id, first_name); city(city_id, city, country_id)'
    Returns {table_name: {col1, col2, ...}}
    """
    tables = {}
    for m in re.finditer(r'(\w+)\s*\(([^)]+)\)', schema):
        tname = m.group(1).lower()
        cols  = {c.strip().lower().split()[0] for c in m.group(2).split(',')}
        cols.discard('')
        tables[tname] = cols
    return tables


# ================================================================
# INTENT SIGNALS
# ================================================================
def question_signals(question):
    q = question.lower()
    return {
        "wants_order": any(w in q for w in [
            "top", "bottom", "highest", "lowest", "most", "least",
            "sort", "order", "rank", "best", "worst", "first", "last",
            "recent", "latest", "earliest", "oldest", "newest",
            "maximum", "minimum", "max", "min",
        ]),
        "wants_group": any(w in q for w in [
            "per", "each", "by", "group", "grouped", "every",
            "breakdown", "split", "average", "avg",
            "sum", "total", "distribution", "count", "how many",
        ]),
        "wants_limit": (
            any(w in q for w in [
                "top", "bottom", "first", "last", "only", "limit",
                "most", "least", "best", "worst",
            ])
            or bool(re.search(r'\b\d+\b', q))
        ),
        "wants_date": any(w in q for w in [
            "date", "day", "week", "month", "year", "today", "yesterday",
            "recent", "latest", "between", "since", "before", "after",
            "this year", "last year", "period", "range", "updated",
            "created", "modified", "last_update", "hire_date",
            "payment_date", "rental_date",
        ]),
        "wants_where": any(w in q for w in [
            "where", "filter", "only", "with", "above", "below",
            "greater", "less", "more than", "fewer", "equal",
            "specific", "that", "who", "which", "whose",
            "in", "not in", "contain", "like", "between",
            "start", "starts", "begin", "begins", "end", "ends",
            "active", "inactive", "status", "named", "called",
        ]),
        "wants_join": any(w in q for w in [
            "join", "related", "linked", "combined",
            "across", "match", "merge", "from both", "with their",
            "and their", "along with",
        ]),
    }


# ================================================================
# FREE-TEXT / NATURAL LANGUAGE DETECTOR
# ================================================================
def is_natural_language(text):
    stripped = text.strip()

    if not re.match(r'^(SELECT|WITH|INSERT|UPDATE|DELETE|CREATE|EXPLAIN|SHOW)', stripped, re.IGNORECASE):
        return True

    if re.search(r'[A-Z][^.!?]{20,}[.!?]\s+[A-Z]', stripped):
        return True

    nl_lines = sum(1 for l in stripped.split('\n') if re.match(r'^\d+\.\s+[A-Za-z]', l.strip()))
    if nl_lines > 1:
        return True

    _NL_INTERIOR = [
        r'(?i)\bplease\s+(include|note|use|see|add)\b',
        r'(?i)\b(as you can see|this query|the result|returns all|which means)\b',
        r'(?i)\b(in order to|you can use|will give you|note that)\b',
        r'(?i)\buse the following schema\b',
        r'(?i)\bthe above (query|sql|statement)\b',
    ]
    for pat in _NL_INTERIOR:
        if re.search(pat, stripped):
            return True

    if re.match(r'^SELECT\b', stripped, re.IGNORECASE):
        has_from = bool(re.search(r'\bFROM\s+\w+', stripped, re.IGNORECASE))
        has_agg_only = bool(re.match(r'^SELECT\s+(COUNT|SUM|AVG|MAX|MIN|NOW|CURRENT)', stripped, re.IGNORECASE))
        if not has_from and not has_agg_only and len(stripped) > 40:
            return True

    return False


# ================================================================
# STAGE 1: SYNTAX VALIDATOR
# ================================================================
_ALWAYS_VALID = {
    "*", "count", "sum", "avg", "max", "min", "coalesce", "nullif",
    "date", "now", "current_date", "current_timestamp", "interval",
    "year", "month", "day", "substring", "length", "upper", "lower",
    "trim", "cast", "extract", "date_trunc", "date_format", "isnull",
    "letter", "total", "cnt", "n", "c", "id", "rank", "row_number",
    "lag", "lead", "dense_rank", "over", "partition",
}

_GARBAGE_PATTERNS = [
    r'^\s*actor_\d',
    r'^\s*\[SQL\]',
    r'^\s*\[EXPLAIN\]',
    r'SELECT\s+actor_\d',
    r'=\s*actor_\d',
    r'City\s+Code\s+Country',
    r'District\s+Code',
    r'(?i)^show\s+\w+[-_]\w+\s+links',
    r'(?i)^show\s+\w+\s+(and|with)\s+\w+\s+(id|links|likes)',
]

def validate_syntax(sql):
    sql = sql.strip()
    if not sql:
        return False, "Empty SQL output."

    if is_natural_language(sql):
        return False, (
            "Output is natural language, not SQL. "
            "Output ONLY a valid SQL statement starting with SELECT (or WITH/INSERT/UPDATE/DELETE). "
            "Do not explain or add any text."
        )

    if not re.match(
        r'^(SELECT|INSERT|UPDATE|DELETE|WITH|CREATE|DROP|EXPLAIN|SHOW)',
        sql, re.IGNORECASE
    ):
        return False, f"Output is not valid SQL: '{sql[:80]}'"

    for pat in _GARBAGE_PATTERNS:
        if re.search(pat, sql, re.IGNORECASE):
            return False, (
                f"Output contains garbage tokens: '{sql[:80]}'. "
                "Output ONLY a valid SQL query."
            )

    if not HAS_SQLGLOT:
        return True, ""
    try:
        tree = sqlglot.parse_one(sql, error_level=sqlglot.ErrorLevel.RAISE)
        if tree is None:
            return False, "SQL parsed to None — likely malformed."
        return True, ""
    except sqlglot.errors.ParseError as e:
        msg = str(e)[:200]
        if any(k in msg.lower() for k in ("unexpected", "invalid identifier")):
            return False, f"Syntax error: {msg}"
        return True, ""
    except Exception:
        return True, ""


# ================================================================
# STAGE 2: SCHEMA VALIDATOR  (multi-table aware + JOIN hints)
# ================================================================
def validate_schema(sql, schema):
    if not HAS_SQLGLOT:
        return True, ""
    tables = parse_schema(schema)
    if not tables:
        return True, ""
    all_tables = set(tables.keys())
    all_cols   = {c for cols in tables.values() for c in cols}
    multi      = len(tables) > 1
    try:
        tree = sqlglot.parse_one(sql)
    except Exception:
        return True, ""

    cte_aliases = {cte.alias.lower() for cte in tree.find_all(exp.CTE)}

    violations = []
    for tbl_node in tree.find_all(exp.Table):
        used = tbl_node.name.lower()
        if not used or used in all_tables or used in _ALWAYS_VALID or used in cte_aliases:
            continue
        if len(used) > 2 and used.isidentifier():
            hint = ""
            if multi:
                fk_hints = _build_join_hints(tables)
                hint = f" Hint — use JOINs: {fk_hints}" if fk_hints else ""
            violations.append(
                f"Table '{used}' not in schema. "
                f"Known tables: {', '.join(sorted(all_tables))}.{hint}"
            )

    if not multi:
        for col_node in tree.find_all(exp.Column):
            used = col_node.name.lower()
            if not used or used == "*" or len(used) <= 2:
                continue
            if used in all_cols or used in _ALWAYS_VALID:
                continue
            violations.append(
                f"Column '{used}' not in schema. "
                f"Valid columns: {', '.join(sorted(all_cols))}."
            )

    table_violations = [v for v in violations if "Table '" in v]
    if table_violations:
        return False, " | ".join(table_violations)
    return True, ""


def _build_join_hints(tables):
    hints = []
    for tbl, cols in tables.items():
        for col in cols:
            if col.endswith('_id') and col != f"{tbl}_id":
                ref = col[:-3]
                if ref in tables:
                    hints.append(f"{tbl} JOIN {ref} ON {tbl}.{col} = {ref}.{ref}_id")
    return "; ".join(hints[:3])


# ================================================================
# STAGE 3: SEMANTIC VALIDATOR
# ================================================================
def validate_semantics(sql, question):
    if not HAS_SQLGLOT:
        return True, ""
    try:
        tree = sqlglot.parse_one(sql)
    except Exception:
        return True, ""
    sig = question_signals(question)
    q   = question.lower()
    violations = []

    is_dml = isinstance(tree, (exp.Update, exp.Delete, exp.Insert))
    is_select_question = not any(w in q for w in [
        "update", "insert", "delete", "remove", "add", "set ",
        "change", "modify", "create", "drop",
    ])
    if is_dml and is_select_question:
        violations.append(
            f"Question asks for data but SQL is a {type(tree).__name__}. "
            "Generate a SELECT statement instead."
        )

    if not is_dml:
        limit_node = tree.find(exp.Limit)
        if limit_node and not sig["wants_limit"] and not sig["wants_order"]:
            limit_sql = limit_node.sql()
            if re.search(r'LIMIT\s+[2-9]\d*', limit_sql, re.IGNORECASE):
                violations.append("Remove LIMIT — question does not specify a number of rows.")

    if violations:
        return False, " | ".join(violations)
    return True, ""


# ================================================================
# COMBINED VALIDATOR
# ================================================================
try:
    import sqlglot
    import sqlglot.expressions as exp
    HAS_SQLGLOT = True
except ImportError:
    HAS_SQLGLOT = False
    print("sqlglot not installed — syntax/schema checks disabled")

def validate_sql(sql, schema, question):
    """Full pipeline: syntax (+ free-text guard) -> schema -> semantics."""
    ok, err = validate_syntax(sql)
    if not ok:
        return False, "SYNTAX", err
    ok, err = validate_schema(sql, schema)
    if not ok:
        return False, "SCHEMA", err
    ok, err = validate_semantics(sql, question)
    if not ok:
        return False, "SEMANTIC", err
    return True, "PASS", ""


print("Validator ready")
print("  parse_schema()        - multi-table aware")
print("  is_natural_language() - free-text / NL guard")
print("  validate_syntax()     - garbage-token + NL detection")
print("  validate_schema()     - multi-table + JOIN hints")
print("  validate_semantics()  - DML/LIMIT checks")
print("  validate_sql()        - full 3-stage pipeline")

## Generate SQL (Validate + Auto-Correct)

In [ ]:
# ================================================================
# GENERATE + VALIDATE + AUTO-CORRECT PIPELINE  (Qwen2.5)
# ================================================================

_GENERATE_BLACKLIST = {"token_type_ids"}

SYSTEM_PROMPT = (
    "You are a strict SQL generator.\n"
    "You must output ONLY valid SQL.\n"
    "Do not explain.\n"
    "Do not add text."
)


def build_prompt(schema, question, correction_hint=""):
    """
    Qwen2.5 chat-template prompt.
    Matches the format used in train_qwen.jsonl.
    For corrections, appends the error to the user turn.
    """
    if correction_hint:
        user_content = (
            f"Use the schema below to write SQL.\n"
            f"Schema: {schema} Question: {question}\n\n"
            f"Your previous SQL was INCORRECT. Fix this specific issue:\n"
            f"{correction_hint}\n"
            "Output ONLY the corrected SQL."
        )
    else:
        user_content = (
            f"Use the schema below to write SQL.\n"
            f"Schema: {schema} Question: {question}"
        )

    # Qwen2.5 chat template — end with assistant header to prime generation
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_content}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


def _run_model(schema, question, correction_hint="", max_new_tokens=200):
    """Single model inference call. Returns cleaned SQL string."""
    prompt = build_prompt(schema, question, correction_hint)
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN)

    for key in _GENERATE_BLACKLIST:
        enc.pop(key, None)
    enc = {k: v.to("cuda") for k, v in enc.items()}

    with torch.no_grad():
        outputs = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            repetition_penalty=1.15,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = outputs[0][enc["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Clean up: strip everything after the first semicolon (keep it)
    if ";" in raw:
        raw = raw[:raw.index(";") + 1]

    # Strip any leaked Qwen chat tokens
    for stop in ["<|im_start|>", "<|im_end|>", "<|endoftext|>", "\n\n"]:
        if stop in raw:
            raw = raw[:raw.index(stop)]

    return raw.strip()


def generate_sql(schema, question, max_retries=4, verbose=False):
    """
    Full pipeline: Generate → Validate → Auto-Correct (up to max_retries).

    Returns dict:
        sql       : final SQL string (best attempt)
        valid     : bool
        attempts  : number of attempts made
        error     : last error string (empty if valid)
        stage     : last failing stage (empty if valid)
    """
    correction_hint = ""
    last_sql, last_error, last_stage = "", "", ""

    for attempt in range(1, max_retries + 1):
        sql = _run_model(schema, question, correction_hint)
        valid, stage, error = validate_sql(sql, schema, question)

        if verbose:
            tag = "✅" if valid else "❌"
            print(f"  Attempt {attempt}: {tag} [{stage}] {sql[:120]}")
            if not valid:
                print(f"           Error: {error[:120]}")

        if valid:
            return {"sql": sql, "valid": True, "attempts": attempt, "error": "", "stage": "PASS"}

        if stage == "SYNTAX":
            correction_hint = f"SYNTAX ERROR: {error}"
        elif stage == "SCHEMA":
            correction_hint = f"SCHEMA ERROR: {error}"
        elif stage == "SEMANTIC":
            correction_hint = f"SEMANTIC ERROR: {error}"
        else:
            correction_hint = error

        last_sql, last_error, last_stage = sql, error, stage

    return {
        "sql": last_sql,
        "valid": False,
        "attempts": max_retries,
        "error": last_error,
        "stage": last_stage,
    }


print("generate_sql() ready")
print(f"  max_retries default : 4")
print(f"  max_new_tokens      : 200")
print(f"  prompt format       : Qwen2.5 chat template")

In [ ]:
# ── Test suite ──
examples = [
    # ── Basic single-table ──────────────────────────────────────────
    {"schema": "orders(id, customer_id, amount, order_date)",
     "question": "Find total revenue"},
    {"schema": "employees(id, name, department, salary, hire_date)",
     "question": "Get the top 5 highest paid employees"},

    # ── LIKE / starts-with ──────────────────────────────────────────
    {"schema": "actor(actor_id, first_name, last_name, last_update)",
     "question": "Select actors whose names start with A"},
    {"schema": "city(city_id, city, country_id, last_update)",
     "question": "Select cities whose name starts with A"},

    # ── Multi-table JOINs ────────────────────────────────────────────
    {"schema": "city(city_id, city, country_id); country(country_id, country)",
     "question": "Show city names along with their country"},
    {"schema": "actor(actor_id, first_name, last_name); film_actor(actor_id, film_id); film(film_id, title)",
     "question": "Select actors and the films they have acted in"},

    # ── Complex: HAVING ─────────────────────────────────────────────
    {"schema": "payment(payment_id, customer_id, amount)",
     "question": "Find customers with total payments over 100"},

    # ── Complex: Subquery ────────────────────────────────────────────
    {"schema": "orders(order_id, customer_id, amount, status)",
     "question": "Find orders above the average order amount"},

    # ── Complex: CASE WHEN ───────────────────────────────────────────
    {"schema": "film(film_id, title, rental_rate)",
     "question": "Classify films as cheap medium or expensive based on rental rate"},

    # ── NULL handling ────────────────────────────────────────────────
    {"schema": "rental(rental_id, customer_id, rental_date, return_date)",
     "question": "Find rentals that were never returned"},
]

print("=" * 70)
passed, failed = 0, 0
for ex in examples:
    print(f"\nSchema   : {ex['schema'][:80]}")
    print(f"Question : {ex['question']}")
    result = generate_sql(schema=ex["schema"], question=ex["question"], verbose=True)
    icon = "✅ OK" if result["valid"] else "⚠️  WARN"
    print(f"[{icon}] Final SQL ({result['attempts']} attempt(s)): {result['sql']}")
    if not result["valid"]:
        print(f"    Still invalid: {result['error']}")
        failed += 1
    else:
        passed += 1
    print("-" * 70)

print(f"\n{'='*70}")
print(f"Results: {passed} passed / {failed} failed out of {len(examples)} tests")

In [ ]:
# ── Your own query — edit schema and question below ──
MY_SCHEMA   = "actor(actor_id, first_name, last_name, last_update)"  # <- edit
MY_QUESTION = "Show actors whose last name starts with S"             # <- edit

result = generate_sql(schema=MY_SCHEMA, question=MY_QUESTION, verbose=True)
print(f"Final SQL : {result['sql']}")
print(f"Valid     : {result['valid']}")
if not result['valid']:
    print(f"Error     : {result['error']}")